In [ ]:
import numpy as np
import pandas as pd
import json
import time
import warnings
warnings.filterwarnings("ignore")

# ============================
# ML Libraries
# ============================
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, make_scorer

import xgboost as xgb
import lightgbm as lgb

import torch
from pytorch_tabnet.tab_model import TabNetRegressor

# ============================
# SETTINGS
# ============================
RANDOM_STATE = 42
N_SPLITS = 5
N_ITER = 30   # safe and acceptable for Q1
TARGET_COL = "yield"

TRAIN_PATH = "E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_2006-24.csv"

# ============================
# LOAD DATA
# ============================
print("Loading training data...")
df = pd.read_csv(TRAIN_PATH)

assert TARGET_COL in df.columns, "Target column missing!"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values

# Scale once (important for TabNet fairness)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Data shape: {X.shape}")

# ============================
# COMMON UTILITIES
# ============================
rmse_scorer = make_scorer(
    lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    greater_is_better=False
)

cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

best_params = {}

# =====================================================
# 1. XGBOOST OPTIMIZATION
# =====================================================
print("\nOptimizing XGBoost...")

xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_param_grid = {
    "n_estimators": [200, 300, 400],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "reg_alpha": [0.0, 0.1, 0.5],
    "reg_lambda": [1.0, 1.5, 2.0]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_grid,
    n_iter=N_ITER,
    scoring=rmse_scorer,
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE
)

start = time.time()
xgb_search.fit(X_scaled, y)
print(f"XGBoost tuning time: {(time.time()-start)/60:.2f} min")

best_params["XGBoost"] = xgb_search.best_params_
print("Best XGBoost params:", best_params["XGBoost"])

# =====================================================
# 2. LIGHTGBM OPTIMIZATION
# =====================================================
print("\nOptimizing LightGBM...")

lgb_model = lgb.LGBMRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

lgb_param_grid = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.03, 0.05, 0.1],
    "num_leaves": [31, 63, 127],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "reg_alpha": [0.0, 0.1, 0.5],
    "reg_lambda": [1.0, 1.5, 2.0]
}

lgb_search = RandomizedSearchCV(
    estimator=lgb_model,
    param_distributions=lgb_param_grid,
    n_iter=N_ITER,
    scoring=rmse_scorer,
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE
)

start = time.time()
lgb_search.fit(X_scaled, y)
print(f"LightGBM tuning time: {(time.time()-start)/60:.2f} min")

best_params["LightGBM"] = lgb_search.best_params_
print("Best LightGBM params:", best_params["LightGBM"])

# =====================================================
# 3. TABNET OPTIMIZATION (CONTROLLED GRID)
# =====================================================
print("\nOptimizing TabNet (controlled search)...")

tabnet_results = []
tabnet_grid = [
    {"n_d": 16, "n_a": 16, "n_steps": 3, "lr": 0.02},
    {"n_d": 32, "n_a": 32, "n_steps": 3, "lr": 0.02},
    {"n_d": 32, "n_a": 32, "n_steps": 4, "lr": 0.01},
]

for cfg in tabnet_grid:
    rmses = []
    for train_idx, val_idx in cv.split(X_scaled):
        X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = TabNetRegressor(
            n_d=cfg["n_d"],
            n_a=cfg["n_a"],
            n_steps=cfg["n_steps"],
            gamma=1.3,
            lambda_sparse=1e-3,
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=cfg["lr"]),
            verbose=0,
            seed=RANDOM_STATE
        )

        model.fit(
            X_tr, y_tr.reshape(-1, 1),
            eval_set=[(X_val, y_val.reshape(-1, 1))],
            max_epochs=80,
            patience=10,
            batch_size=512,
            virtual_batch_size=128,
            eval_metric=["rmse"]
        )

        preds = model.predict(X_val).flatten()
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmses.append(rmse)

    tabnet_results.append((cfg, np.mean(rmses)))

best_tabnet = sorted(tabnet_results, key=lambda x: x[1])[0]
best_params["TabNet"] = best_tabnet[0]

print("Best TabNet params:", best_params["TabNet"])

# =====================================================
# SAVE RESULTS
# =====================================================
with open("best_hyperparameters.json", "w") as f:
    json.dump(best_params, f, indent=4)

print("\n Hyperparameter optimization completed.")
print(" Saved to best_hyperparameters.json")


Loading training data...
Data shape: (12372, 127)

Optimizing XGBoost...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
XGBoost tuning time: 27.74 min
Best XGBoost params: {'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.9}

Optimizing LightGBM...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012370 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 31904
[LightGBM] [Info] Number of data points in the train set: 9897, number of used features: 127
[LightGBM] [Info] Start training from score 152.283722
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018319 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 31903
[LightGBM] [Info] Number of data points in t